# Random Forest — 1. Bagging

Since **Bagging** is the first subtopic, let's start there.

## 1. What is Bagging?

**Bagging = Bootstrap Aggregating.**

It is an ensemble technique where we:

1. Create multiple different training datasets from the original dataset.
2. Train a separate model on each dataset.
3. Combine their predictions.

For Random Forest, those models are **Decision Trees**.

So:

```text
Original Dataset
       │
       ├── Bootstrap Dataset 1 → Decision Tree 1 ──┐
       ├── Bootstrap Dataset 2 → Decision Tree 2 ──┤
       ├── Bootstrap Dataset 3 → Decision Tree 3 ──┤
       ├── Bootstrap Dataset 4 → Decision Tree 4 ──┤
       └── Bootstrap Dataset 5 → Decision Tree 5 ──┘
                                                    │
                                             Combine predictions
                                                    │
                                                Final prediction
```

---

## 2. Why do we need Bagging?

A single Decision Tree can have **high variance**.

Small changes in the training data can produce a significantly different tree.

Bagging reduces this instability by training many trees on different versions of the dataset and combining them.

The important idea is:

> **Individual trees may be unstable, but their combined prediction is more stable.**

---

## 3. Bootstrap Sampling

This is how Bagging creates different datasets.

Suppose our original dataset has 5 samples:

```text
A B C D E
```

A bootstrap sample of size 5 is created by **random sampling with replacement**.

For example:

```text
A C C E B
```

Notice:

- `C` appears twice.
- `D` doesn't appear.
- The sample still contains exactly 5 observations.

Another tree might receive:

```text
B B D A E
```

Another:

```text
C D D E A
```

So every tree sees a different training dataset.

---

## 4. What does "with replacement" mean?

After selecting a sample, we **put it back** before selecting the next one.

For example:

```text
Original: A B C D E

Pick C
→ C is returned

Pick C again
→ C can be selected again

Pick A
→ A is returned

Pick E
→ E is returned
```

Therefore duplicates are possible.

---

## 5. Combining Predictions

For **classification**, Random Forest uses majority voting.

Suppose 5 trees predict:

```text
Tree 1 → Cat
Tree 2 → Dog
Tree 3 → Cat
Tree 4 → Cat
Tree 5 → Dog
```

Votes:

```text
Cat → 3
Dog → 2
```

Final prediction:

```text
Cat
```

For **regression**, predictions are averaged:

$$
\hat{y} = \frac{1}{B}\sum_{b=1}^{B}\hat{y}_b
$$

where:

- $B$ = number of trees
- $\hat{y}_b$ = prediction from tree $b$
- $\hat{y}$ = final prediction

---

## 6. Bagging vs Random Forest

This distinction is important.

**Bagging Decision Trees:**

- Different bootstrap samples
- Each tree can potentially consider **all features**

**Random Forest:**

- Different bootstrap samples
- Additionally, each split considers only a **random subset of features**

So Random Forest is essentially:

> **Bagging + Random Feature Selection + Decision Trees**


### Bagging

If the original dataset has **100 rows and 10 features**:

- Bootstrap sampling selects **100 rows with replacement** for each tree.
- So each bootstrap dataset has **100 rows**.
- The rows can repeat, and some original rows may not appear.
- Different trees get different bootstrap samples.
- **When a tree makes a split, it can consider all 10 features.**

```text
Original: 100 rows × 10 features

Bootstrap 1 → 100 rows × 10 features → Tree 1
Bootstrap 2 → 100 rows × 10 features → Tree 2
Bootstrap 3 → 100 rows × 10 features → Tree 3
```

The **number of rows stays the same**, but their composition can differ.

---

### Random Forest

Random Forest also does the same bootstrap sampling:

```text
Bootstrap 1 → 100 rows → Tree 1
Bootstrap 2 → 100 rows → Tree 2
...
```

But now, when growing each tree:

**At every split**, instead of considering all 10 features, it randomly selects a subset.

For example:

```text
10 total features

Split 1 → randomly consider F2, F4, F7
Split 2 → randomly consider F1, F5, F9
Split 3 → randomly consider F3, F6, F10
```

Then the tree finds the **best split among only those selected features**.

### So your statement in one line:

> **Bagging:** randomizes the **rows** using bootstrap sampling, while the tree can consider all features.

> **Random Forest:** randomizes the **rows + features**; the feature subset is selected **at each split**.

And yes, if the original dataset has **100 rows**, every bootstrap sample is also **100 rows**. It is **not necessary for the bootstrap samples of different trees to contain the same rows**.

## Random Forest — 2. Bootstrap Sampling

Now let's study **Bootstrap Sampling** specifically, because this is the mechanism that creates different datasets for the different trees.

### What is Bootstrap Sampling?

Bootstrap sampling means:

> Randomly select samples from the original dataset **with replacement**, until you have the same number of samples as the original dataset.

Suppose:

```text
Original dataset = 5 rows

A B C D E
```

We need to create a bootstrap dataset of **5 rows**.

One possible result:

```text
A C C E B
```

Notice:

- `C` appears twice.
- `D` doesn't appear.
- Total rows = 5, same as original.

Another tree could receive:

```text
D D A E C
```

So every tree receives a different version of the training data.

---

### Why "with replacement"?

After selecting a row, we put it back into the pool before selecting the next row.

For example:

```text
Pool: A B C D E

Pick C → C is returned
Pick C → C can be picked again
Pick A → A is returned
Pick E → E is returned
Pick B
```

Therefore, duplicates are possible.

---

### Important point

If original dataset has **N rows**:

$$
\boxed{\text{Bootstrap sample size} = N}
$$

But:

$$
\boxed{\text{Unique rows in bootstrap sample} < N \text{ usually}}
$$

because some rows repeat.

This is what gives different trees different training data and contributes to the diversity of the Random Forest.

## Random Forest — 3. Random Feature Selection

Now we move to the second source of randomness in Random Forest.

### What is Random Feature Selection?

At **each split of each decision tree**, Random Forest randomly selects a subset of the available features.

The tree then searches for the **best split only among those selected features**.

---

### Example

Suppose we have **10 features**:

```text
F1 F2 F3 F4 F5 F6 F7 F8 F9 F10
```

At the root node, Random Forest might randomly select:

```text
F2 F5 F7
```

The tree evaluates possible splits using **only F2, F5, and F7**.

It might decide:

```text
F5 < 20
```

Then we move to the next node.

At that node, a **new random subset** is selected:

```text
F1 F4 F9
```

The tree now searches for the best split only among these.

Then at another node:

```text
F3 F6 F8
```

Again, a new subset.

---

### Important: It's at every split

It is **not**:

```text
Tree 1 → permanently use F1,F4,F7
```

Instead:

```text
Tree
 │
 ├── Split 1 → random subset → choose best split
 │
 ├── Split 2 → new random subset → choose best split
 │
 ├── Split 3 → new random subset → choose best split
 │
 └── ...
```

This randomness makes trees less correlated with each other.

### Why?

If every tree always had access to all features, strong features could dominate many trees.

By restricting the features considered at each split, different trees can learn different structures.

So Random Forest gets diversity from **both**:

$$
\boxed{\text{Bootstrap Sampling} + \text{Random Feature Selection}}
$$

The number of features selected at each split is controlled by the `max_features` parameter in implementations such as scikit-learn.

## Random Forest — 4. Out-of-Bag (OOB) Evaluation

OOB evaluation is another important concept that comes naturally from **bootstrap sampling**.

### The key idea

Remember:

If the original dataset has **N rows**, each tree gets **N random selections with replacement**.

Because sampling is with replacement, some original rows will **not be selected at all** for a particular tree.

Those unused rows are called:

> **Out-of-Bag (OOB) samples**

### Example

Original dataset:

```text
A B C D E
```

Bootstrap sample for Tree 1:

```text
A C C E B
```

Which original sample was never selected?

```text
D
```

Therefore:

```text
D → OOB sample for Tree 1
```

Another tree might have:

```text
B B D A E
```

Here:

```text
C → OOB sample for Tree 2
```

So **different trees have different OOB samples**.

---

### How is OOB evaluation used?

Suppose sample `D` was not used to train Tree 1.

We can now give `D` to Tree 1 and ask:

```text
What does Tree 1 predict for D?
```

Because Tree 1 **never trained on D**, this prediction can be used to evaluate that tree on D.

Across the entire Random Forest, each training sample is usually OOB for many trees.

For a particular sample:

```text
Sample X
   │
   ├── Tree 1 → OOB → prediction ✓
   ├── Tree 2 → trained on X → ignore
   ├── Tree 3 → OOB → prediction ✓
   ├── Tree 4 → trained on X → ignore
   └── Tree 5 → OOB → prediction ✓
```

We combine only the predictions from the trees for which that sample was OOB.

For classification:

**majority vote → OOB prediction**

For regression:

**average → OOB prediction**

Then compare the OOB prediction with the actual target.

---

### Why is this useful?

Normally, to evaluate a model properly, we might keep a separate validation/test set.

OOB evaluation gives Random Forest an **internal estimate of generalization performance** using samples that individual trees did not train on.

So the important distinction is:

```text
Training sample for a tree
        ↓
Used to train that tree

OOB sample for that tree
        ↓
Not used to train that tree
        ↓
Can be used to evaluate that tree
```

The `oob_score` option in scikit-learn enables this type of evaluation.

### OOB = Out-of-Bag samples

Suppose your original dataset has:

```text
100 rows × 10 features
```

For Tree 1, bootstrap sampling creates:

```text
100 selected rows
```

Because sampling is **with replacement**, maybe only 63 unique rows were actually selected.

The remaining ~37 rows are **OOB samples for Tree 1**.

```text
Original 100 rows
        │
        ├── Used to train Tree 1 → ~63 unique rows
        │
        └── NOT used to train Tree 1 → ~37 OOB rows
```

Those OOB **rows** are then used to evaluate the tree.

### Don't confuse these:

| Concept | What is randomized? |
|---|---|
| **Bootstrap sampling** | Rows/samples |
| **OOB samples** | Rows not selected for a particular tree |
| **Random feature selection** | Features considered at each split |

So:

> **OOB score = performance evaluated using training samples that were not used to train the particular trees being evaluated.**

It has **nothing to do with features being excluded from a tree's training**.

## Random Forest — 3. Random Feature Selection

We already discussed the concept, so let's now understand it more precisely.

### How many features are selected?

Suppose your dataset has:

```text
10 features
```

At a particular node, Random Forest might select only:

```text
3 features
```

Then it searches for the best split **only among those 3 features**.

For example:

```text
All features:
F1 F2 F3 F4 F5 F6 F7 F8 F9 F10
             ↓
Randomly selected:
F2 F5 F8
             ↓
Test possible splits
             ↓
Choose best split
```

The important point is that the **number of features selected is controlled by `max_features`**.

For example, conceptually:

```python
RandomForestClassifier(max_features=3)
```

means that at each split, up to **3 randomly selected features** are considered.

---

### What happens at the next node?

A **new random subset** is selected.

```text
Root node:
F2 F5 F8 → choose best split

        ↓

Child node:
F1 F4 F9 → choose best split

        ↓

Next node:
F3 F6 F10 → choose best split
```

So the feature subset isn't fixed for the entire tree.

### Why?

It forces different trees to explore different combinations of features, making the trees **less correlated**.

And this is one of the main reasons Random Forest generally performs better than simply averaging many highly similar Decision Trees.

## Random Forest — 4. OOB Evaluation

We already touched OOB earlier, so let's now understand **how the OOB score is actually calculated**.

### Step 1 — Bootstrap samples

Suppose we have 5 trees:

```text
Tree 1 → trained on its bootstrap sample
Tree 2 → trained on its bootstrap sample
Tree 3 → trained on its bootstrap sample
Tree 4 → trained on its bootstrap sample
Tree 5 → trained on its bootstrap sample
```

Because each tree samples **with replacement**, every tree leaves some rows unused.

---

### Step 2 — Find OOB trees for a sample

Suppose sample `X` was:

```text
Tree 1 → OOB
Tree 2 → Used for training
Tree 3 → OOB
Tree 4 → OOB
Tree 5 → Used for training
```

Only the OOB trees are allowed to predict `X`:

```text
Tree 1 → Class A
Tree 3 → Class B
Tree 4 → Class A
```

Majority vote:

```text
Class A → 2 votes
Class B → 1 vote

OOB prediction → Class A
```

Then compare that with the actual class of `X`.

---

### Step 3 — Repeat for every training sample

Every training sample gets an OOB prediction using **only the trees that did not train on that sample**.

Then:

$$
\text{OOB Score}
=
\frac{\text{Correct OOB Predictions}}
{\text{Total Samples}}
$$

For classification, this is essentially the **accuracy** of the OOB predictions.

For example:

```text
100 training samples
85 correct OOB predictions

OOB Score = 85 / 100 = 0.85
```

So:

**OOB Score = 85%**

The important point is that the model is being evaluated on training samples that were **left out of the bootstrap sample for the trees making those predictions**.

Suppose we have **3 trees** and 5 training rows:

```text
A  B  C  D  E
```

### Step 1 — Bootstrap samples

Each tree gets its own bootstrap sample:

```text
Tree 1 → A B B D E
Tree 2 → A C D D E
Tree 3 → B C C D A
```

Now look at row **C**.

```text
Tree 1 → C was NOT selected → C is OOB for Tree 1
Tree 2 → C was selected     → C was used for training
Tree 3 → C was selected     → C was used for training
```

So **C is OOB only for Tree 1**.

---

### Step 2 — Give C to Tree 1

Now we take the **same row C** and pass it through Tree 1:

```text
C
 ↓
Tree 1
 ↓
prediction = Class 0
```

Why are we allowed to do this?

Because Tree 1 **never saw C during its training**.

So Tree 1 is effectively being tested on C.

---

### Step 3 — Suppose C is OOB for multiple trees

Imagine we have 100 trees.

For row C:

```text
Tree 1  → OOB → predicts Class 0
Tree 7  → OOB → predicts Class 1
Tree 15 → OOB → predicts Class 0
Tree 31 → OOB → predicts Class 0
Tree 42 → OOB → predicts Class 1
...
```

We **only take predictions from the trees where C was OOB**.

Then for classification:

```text
Class 0 → 3 votes
Class 1 → 2 votes

Final OOB prediction for C = Class 0
```

Then we compare:

```text
Actual class of C = Class 0
OOB prediction      = Class 0
                         ↓
                      Correct
```

---

### The important distinction

When you normally give a **new unseen input** to Random Forest:

```text
New input
   ↓
ALL trees
   ↓
predictions
   ↓
majority vote
```

But for **OOB evaluation**:

```text
Training row C
      ↓
Only trees that DID NOT train on C
      ↓
OOB predictions
      ↓
majority vote
      ↓
compare with C's actual label
```

So **OOB is not a different prediction mechanism for new data**.

It is a way of getting an **internal test-like prediction for the training rows**, using only trees that didn't see those rows during training.

## Random Forest — 5. Hyperparameters

Now we'll start the hyperparameters of Random Forest.

The most important ones are:

- `n_estimators`
- `max_depth`
- `min_samples_split`
- `min_samples_leaf`
- `max_features`
- `bootstrap`
- `criterion`
- `max_leaf_nodes`
- `oob_score`
- `random_state`

We'll take them **one at a time**.

### 1. `n_estimators`

This controls:

> **How many Decision Trees are created in the Random Forest.**

Example:

```python
RandomForestClassifier(n_estimators=100)
```

means:

```text
100 Decision Trees
       ↓
Predictions from all trees
       ↓
Majority voting
       ↓
Final prediction
```

If:

```python
n_estimators=10
```

→ 10 trees.

If:

```python
n_estimators=500
```

→ 500 trees.

### What happens when we increase it?

Generally:

```text
More trees
   ↓
More stable ensemble
   ↓
Usually better generalization
   ↓
More computation/time
```

Unlike `max_depth`, increasing `n_estimators` generally doesn't cause the same kind of overfitting problem; after some point, performance simply tends to **plateau** while computation increases.


### 2. `max_depth`

`max_depth` controls the **maximum depth of every Decision Tree** in the Random Forest.

For example:

```python
RandomForestClassifier(max_depth=5)
```

means each tree can grow to a maximum depth of **5**.

```text
                 Root
                  │ depth 0
             ┌────┴────┐
             │         │ depth 1
           ...        ...
                       │
                    depth 5
                       ↓
                 Cannot split further
```

### Effect of `max_depth`

**Small `max_depth`:**

```text
Shallow trees
     ↓
Less complex
     ↓
May underfit
```

**Large `max_depth`:**

```text
Deeper trees
     ↓
More complex
     ↓
Can capture more patterns
     ↓
Higher overfitting risk
```

For example:

```python
RandomForestClassifier(max_depth=3)
```

→ relatively shallow trees.

```python
RandomForestClassifier(max_depth=None)
```

→ trees can continue growing until another stopping condition is reached.

The important point is that `max_depth` applies **to each individual Decision Tree**, not to the Random Forest as a whole.

### 3. `min_samples_split`

`min_samples_split` specifies the **minimum number of samples a node must have before it is allowed to split**.

For example:

```python
RandomForestClassifier(min_samples_split=10)
```

Suppose a node contains:

```text
20 samples
```

Since:

$$
20 \geq 10
$$

the node **can attempt a split**.

If another node contains:

```text
7 samples
```

then:

$$
7 < 10
$$

so it **cannot split** and becomes a leaf.

### Important distinction

`min_samples_split` checks the **current node before splitting**.

```text
Current node
    │
    ├── 20 samples → can split
    │
    └── 7 samples → cannot split
```

It does **not** check how many samples the resulting children will have. That's the job of `min_samples_leaf`.

### Effect

Higher `min_samples_split`:

```text
Fewer splits
→ simpler trees
→ lower overfitting risk
→ possible underfitting
```

Lower `min_samples_split`:

```text
More splits
→ more complex trees
→ can capture more patterns
→ higher overfitting risk
```

### 4. `min_samples_leaf`

`min_samples_leaf` specifies the **minimum number of samples that must be present in each leaf after a split**.

For example:

```python
RandomForestClassifier(min_samples_leaf=5)
```

Suppose the tree considers this split:

```text
Current node: 20 samples

             Split
            /     \
         17         3
```

This split is **rejected** because the right child has only 3 samples:

```text
3 < 5
```

But:

```text
             Split
            /     \
          12        8
```

is valid because:

```text
12 >= 5
8  >= 5
```

### Difference from `min_samples_split`

| Parameter | Question it asks |
|---|---|
| `min_samples_split` | **Can this current node split?** |
| `min_samples_leaf` | **Will both resulting children have enough samples?** |

So the sequence is:

```text
Current node
     ↓
min_samples_split
     ↓
Can it split?
     ↓
Find a candidate split
     ↓
min_samples_leaf
     ↓
Are both children large enough?
     ↓
Accept / reject split
```

Higher `min_samples_leaf` forces larger leaves and therefore produces **simpler trees**.

### 5. `max_features`

`max_features` controls **how many features are randomly considered at each split**.

Suppose your dataset has **10 features**:

```text
F1 F2 F3 F4 F5 F6 F7 F8 F9 F10
```

If:

```python
RandomForestClassifier(max_features=3)
```

then at each split, Random Forest randomly selects **3 features**.

Example:

```text
Split 1 → F2, F5, F8
             ↓
       find best split

Split 2 → F1, F4, F9
             ↓
       find best split

Split 3 → F3, F6, F10
             ↓
       find best split
```

The tree **does not evaluate all 10 features** at those splits.

### Why is this important?

It increases diversity between trees.

Without feature randomness:

```text
Tree 1 → F2, F5 are repeatedly best
Tree 2 → F2, F5 are repeatedly best
Tree 3 → F2, F5 are repeatedly best
```

The trees can become very similar.

With `max_features`, different trees/splits get different feature subsets, making the trees less correlated.

### Common values

In scikit-learn, `max_features` can be specified in different ways, such as:

```python
max_features=3        # exactly 3 features
max_features="sqrt"   # sqrt(number of features)
max_features="log2"   # log2(number of features)
max_features=1.0      # 100% of features
```

For example, with 16 features:

$$
\sqrt{16}=4
$$

So:

```python
max_features="sqrt"
```

means approximately **4 randomly selected features at each split**.

### 6. `bootstrap`

`bootstrap` controls whether Random Forest uses **bootstrap sampling** to create the training dataset for each tree.

```python
RandomForestClassifier(bootstrap=True)
```

### `bootstrap=True`

This is the standard Random Forest behavior.

Each tree gets a bootstrap sample:

```text
Original:       A B C D E

Tree 1:         A C C E B
Tree 2:         B B D A E
Tree 3:         C D D E A
```

Sampling is **with replacement**, so:

- duplicates can occur
- some original rows are not selected
- those unselected rows become **OOB samples**

---

### `bootstrap=False`

No bootstrap sampling is performed.

Each tree receives the **entire training dataset**:

```text
Tree 1 → A B C D E
Tree 2 → A B C D E
Tree 3 → A B C D E
```

Therefore, the normal OOB mechanism is not available because there are no bootstrap-excluded rows.

### In short

```text
bootstrap=True
    ↓
Bootstrap samples
    ↓
Different rows for different trees
    ↓
OOB evaluation possible
```

```text
bootstrap=False
    ↓
Each tree gets the full dataset
    ↓
No bootstrap-based OOB samples
```

But with `max_features < number_of_features`, each tree can consider different random feature subsets at each split:

```text
Same dataset
     ↓
 ┌───────┬───────┬───────┐
Tree 1  Tree 2  Tree 3  ...
 ↓       ↓       ↓
different random feature choices
```

Therefore, the trees can develop **different structures and predictions**.

### But if you also had:

```python
bootstrap=False
max_features=1.0
```

then every tree sees:

- the same rows
- all the same features
- same splitting criteria

With the same deterministic settings, the trees would generally be **identical**, so there would be little/no benefit from having 10 of them.

That's why Random Forest's two sources of randomness are important:

$$
\boxed{\text{Random rows} + \text{Random features}}
$$

### 7. `criterion`

`criterion` determines **how Random Forest evaluates the quality of a split** in each Decision Tree.

It is the same basic concept we learned for Decision Trees.

For classification, common criteria include:

```python
criterion="gini"
criterion="entropy"
criterion="log_loss"
```

### `gini`

Uses **Gini impurity**:

$$
Gini = 1-\sum_{k=1}^{K}p_k^2
$$

The tree chooses splits that produce a **lower weighted Gini impurity**.

---

### `entropy`

Uses entropy:

$$
H=-\sum_{k=1}^{K}p_k\log_2(p_k)
$$

The tree prefers splits that produce **lower weighted entropy**, or equivalently higher information gain.

---

### Example

Suppose Random Forest has selected these features for a particular split:

```text
F2, F5, F8
```

It tests possible splits:

```text
F2 < 20
F2 < 30
F5 < 10
F5 < 15
F8 < 40
...
```

If:

```python
criterion="gini"
```

it chooses the split with the best **Gini-based score**.

If:

```python
criterion="entropy"
```

it chooses the split with the best **entropy/information-gain-based score**.

So `criterion` **doesn't decide which features are randomly selected**. `max_features` does that.

```text
max_features → Which features can be considered?
criterion     → How do we judge the quality of their splits?
```

### 8. `max_leaf_nodes`

`max_leaf_nodes` controls the **maximum number of leaf nodes a Decision Tree can have** inside each Random Forest tree.

For example:

```python id="w9j5qg"
RandomForestClassifier(max_leaf_nodes=10)
```

means each individual tree can have **at most 10 leaves**.

### Why does this matter?

Consider a tree:

```text id="r8p6sm"
              Root
             /    \
            /      \
           /        \
        Node        Node
       /   \        /   \
      L1   L2      L3   L4
```

There are 4 leaf nodes.

If:

```python id="e2x0sl"
max_leaf_nodes=4
```

the tree cannot grow beyond 4 leaves.

A smaller value:

```text id="7h2w0y"
max_leaf_nodes ↓
       ↓
Fewer leaves
       ↓
Simpler tree
       ↓
Lower complexity
```

A larger value:

```text id="f2n8qc"
max_leaf_nodes ↑
       ↓
More leaves allowed
       ↓
More complex tree
       ↓
Higher overfitting possibility
```

### Difference from `max_depth`

They control complexity in different ways:

- **`max_depth`** → limits how many levels the tree can have.
- **`max_leaf_nodes`** → limits how many final prediction nodes the tree can have.

A tree can have a small depth but many leaves, or greater depth with relatively few leaves depending on its structure.

### 9. `oob_score`

`oob_score` controls whether Random Forest calculates an **Out-of-Bag score** during training.

```python id="y5v8bq"
RandomForestClassifier(
    n_estimators=100,
    oob_score=True
)
```

### `oob_score=False`

OOB evaluation is not calculated.

### `oob_score=True`

Random Forest:

1. Creates bootstrap samples.
2. Trains the trees.
3. For each training row, finds the trees that **didn't train on that row**.
4. Uses those trees to predict that row.
5. Compares the OOB prediction with the actual target.
6. Calculates the overall OOB score.

For classification, it is commonly an accuracy-like score:

$$
OOB\ Score =
\frac{\text{Correct OOB predictions}}
{\text{Total samples}}
$$

For example:

```text id="v0g6a8"
100 training samples
85 correct OOB predictions

OOB score = 0.85
```

So:

```python id="7w4qph"
model.oob_score_
```

could give:

```text id="xj1b7r"
0.85
```

### Important

`oob_score=True` requires **bootstrap sampling**, because OOB samples come from rows that were left out of each tree's bootstrap sample.

So if:

```python id="kqz9xz"
bootstrap=False
oob_score=True
```

you cannot use the normal OOB evaluation mechanism.

Exactly. Let's use a **small example** to see how many OOB predictions become **one score**.

Suppose we have **5 training rows**:

| Row | Actual class |
|---|---|
| A | 0 |
| B | 1 |
| C | 0 |
| D | 1 |
| E | 0 |

And suppose we have 4 trees.

### Step 1 — Get OOB predictions

For each row, we only use trees where that row was OOB.

After combining those trees' predictions, we might get:

| Row | Actual | OOB prediction |
|---|---:|---:|
| A | 0 | 0 |
| B | 1 | 1 |
| C | 0 | 1 |
| D | 1 | 1 |
| E | 0 | 0 |

Notice that each row's prediction may have come from a **different number of trees**.

For example:

```text
A → Tree 1, Tree 3
B → Tree 2, Tree 4
C → Tree 1, Tree 2, Tree 4
...
```

But after voting, each row has **one final OOB prediction**.

---

### Step 2 — Compare predictions with actual values

```text
A → 0 = 0 ✓
B → 1 = 1 ✓
C → 1 ≠ 0 ✗
D → 1 = 1 ✓
E → 0 = 0 ✓
```

So:

```text
Correct = 4
Total = 5
```

Therefore:

$$
OOB\ Score = \frac{4}{5}=0.8
$$

So:

$$
\boxed{OOB\ Score=80\%}
$$

### The whole process

```text
Many trees
    ↓
OOB predictions for each row
    ↓
One prediction per row
    ↓
Compare with actual value
    ↓
Count correct predictions
    ↓
ONE OOB score
```

For **classification**, this is essentially the accuracy of those OOB predictions.

For **regression**, instead of counting correct/incorrect, an appropriate regression metric is calculated from the OOB predictions and actual values.